You can download the `requirements.txt` for this course from the workspace of this lab. `File --> Open...`

# L2: Create Agents to Research and Write an Article

In this lesson, you will be introduced to the foundational concepts of multi-agent systems and get an overview of the crewAI framework.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import from the crewAI libray.

In [2]:
from crewai import Agent, Task, Crew

- As a LLM for your agents, you'll be using OpenAI's `gpt-3.5-turbo`.

**Optional Note:** crewAI also allow other popular models to be used as a LLM for your Agents. You can see some of the examples at the [bottom of the notebook](#1).

In [3]:
import os
from utils import get_openai_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'

## Creating Agents

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

### Agent: Planner

**Note**: The benefit of using _multiple strings_ :
```Python
varname = "line 1 of text"
          "line 2 of text"
```

versus the _triple quote docstring_:
```Python
varname = """line 1 of text
             line 2 of text
          """
```
is that it can avoid adding those whitespaces and newline characters, making it better formatted to be passed to the LLM.

In [4]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    allow_delegation=False,
	verbose=True
)

### Agent: Writer

In [5]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    verbose=True
)

### Agent: Editor

In [6]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    allow_delegation=False,
    verbose=True
)

## Creating Tasks

- Define your Tasks, and provide them a `description`, `expected_output` and `agent`.

### Task: Plan

In [7]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
)

### Task: Write

In [8]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
		"3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.",
    agent=writer,
)

### Task: Edit

In [9]:
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor
)

## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: *For this simple example*, the tasks will be performed sequentially (i.e they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=2` allows you to see all the logs of the execution. 

In [10]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=2
)

## Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [11]:
result = crew.kickoff(inputs={"topic": "Artificial Intelligence"})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on Artificial Intelligence.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
I now can give a great answer

Final Answer:
Content Plan Document:

Title: The Latest Trends and Key Players in Artificial Intelligence

Introduction:
- Brief overview of Artificial Intelligence (AI)
- Importance of staying updated on the latest trends in AI
- Mention of key players in the industry

Key Points:
1. Latest Trends in AI
- Deep learning advancements
- Natural language processing developments
- AI ethics and regulations
- AI-powered automation in various industries

2. Key Players in AI
- Google AI
- IBM Watson
- Amazon AWS
- Microsoft AI

3

I now can give a great answer

Final Answer:
# The Latest Trends and Key Players in Artificial Intelligence

**Introduction:**
Artificial Intelligence (AI) has become a driving force in today's technological landscape, reshaping industries and transforming how we interact with technology. Staying abreast of the latest trends in AI is imperative for technology enthusiasts, business professionals, and students looking to delve into this rapidly evolving field. Key players in the AI industry, including Google AI, IBM Watson, Amazon AWS, and Microsoft AI, are at the forefront of innovation, pushing boundaries in AI research and development.

**Latest Trends in AI:**
Advancements in deep learning have empowered AI systems to glean insights from vast datasets and make intricate decisions, resulting in breakthroughs in areas like image recognition and natural language processing. Progress in natural language processing has also been substantial, enabling AI to comprehend and produce human lan

- Display the results of your execution as markdown in the notebook.

In [12]:
from IPython.display import Markdown
Markdown(result)

# The Latest Trends and Key Players in Artificial Intelligence

**Introduction:**
Artificial Intelligence (AI) has become a driving force in today's technological landscape, reshaping industries and transforming how we interact with technology. Staying abreast of the latest trends in AI is imperative for technology enthusiasts, business professionals, and students looking to delve into this rapidly evolving field. Key players in the AI industry, including Google AI, IBM Watson, Amazon AWS, and Microsoft AI, are at the forefront of innovation, pushing boundaries in AI research and development.

**Latest Trends in AI:**
Advancements in deep learning have empowered AI systems to glean insights from vast datasets and make intricate decisions, resulting in breakthroughs in areas like image recognition and natural language processing. Progress in natural language processing has also been substantial, enabling AI to comprehend and produce human language with heightened accuracy. The emergence of AI ethics and regulations underscores the significance of responsible AI advancement and deployment, ensuring that AI technologies are utilized ethically and transparently. AI-driven automation is revolutionizing sectors such as healthcare, finance, and manufacturing, enhancing efficiency and streamlining processes.

**Key Players in AI:**
Google AI, with its focus on machine learning and AI applications, remains a frontrunner in AI innovation. IBM Watson's cognitive computing capabilities have transformed industries by harnessing AI to analyze extensive data sets and extract valuable insights. Amazon AWS provides a diverse array of AI services, encompassing machine learning and natural language processing, empowering businesses to seamlessly integrate AI into their operations. Microsoft AI is dedicated to democratizing AI technologies, making them more accessible to a wider audience and catalyzing AI adoption across diverse sectors.

**Noteworthy News in AI:**
Recent breakthroughs in AI technology, such as the language generation prowess of GPT-3 and AlphaFold's predictions on protein folding, exemplify the rapid progress in AI research. The impact of AI on society and the economy is undeniable, with AI technologies reshaping industries, creating new job prospects, and sparking ethical debates regarding data privacy and bias. Future projections for AI development include heightened AI capabilities, increased automation, and the integration of AI into everyday life, shaping the trajectory of technology and society.

**Conclusion:**
Remaining well-informed about the latest trends and key players in Artificial Intelligence is imperative for staying ahead in the dynamic technological landscape. By delving into additional resources on AI trends, engaging in dialogues on AI subjects, and partaking in AI-related events and webinars, individuals can deepen their comprehension of AI and its impact on various industries. As AI continues to progress and mold the future of technology, staying informed and involved in AI advancements is vital for individuals and businesses alike.

## Try it Yourself

- Pass in a topic of your choice and see what the agents come up with!

In [13]:
topic = "YOUR TOPIC HERE"
result = crew.kickoff(inputs={"topic": topic})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on YOUR TOPIC HERE.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
I now can give a great answer

Final Answer:

Content Plan for Blog Article on "YOUR TOPIC HERE"

1. Latest Trends, Key Players, and Noteworthy News:
- Trends: 
- Key Players: 
- Noteworthy News: 

2. Target Audience Analysis:
- Demographics: 
- Interests: 
- Pain Points: 

3. Detailed Content Outline:
- Introduction:
- Key Points:
1. 
2. 
3. 
- Call to Action:

4. SEO Keywords and Relevant Data/Sources:
- Keywords:
1. 
2. 
3. 
- Sources:
1. 
2. 
3. 

Please note that the content plan outlined above is subject to further research and revisions before finalizing 

I now can give a great answer

Final Answer:

# The Future of Artificial Intelligence in Healthcare

## Introduction

The rapid advancement of technology in the healthcare industry, especially in the realm of artificial intelligence (AI), has the potential to revolutionize the way healthcare is delivered. From enhancing diagnostics to creating personalized treatment plans, AI is paving the way for a more efficient and effective healthcare system. This blog post delves into the latest trends, key players, and noteworthy news in the field of AI in healthcare.

## Latest Trends, Key Players, and Noteworthy News

### Trends:
A significant trend in AI in healthcare is the utilization of machine learning algorithms to analyze medical images for diagnostic purposes. Companies such as IBM Watson Health and Google Health are leading the way in developing AI-powered tools for radiology and pathology. Another emerging trend is the use of natural language processing to extract valuable information

In [14]:
Markdown(result)

# The Future of Artificial Intelligence in Healthcare

## Introduction

The rapid advancement of technology in the healthcare industry, especially in the realm of artificial intelligence (AI), has the potential to revolutionize the way healthcare is delivered. From enhancing diagnostics to creating personalized treatment plans, AI is paving the way for a more efficient and effective healthcare system. This blog post delves into the latest trends, key players, and noteworthy news in the field of AI in healthcare.

## Latest Trends, Key Players, and Noteworthy News

### Trends:
A significant trend in AI in healthcare is the utilization of machine learning algorithms to analyze medical images for diagnostic purposes. Companies such as IBM Watson Health and Google Health are leading the way in developing AI-powered tools for radiology and pathology. Another emerging trend is the use of natural language processing to extract valuable information from electronic health records, enabling healthcare providers to make more informed decisions.

### Key Players:
Companies like NVIDIA, GE Healthcare, and Siemens Healthineers are key players in the AI healthcare space, heavily investing in AI research and development to introduce innovative solutions to the market. Additionally, startups like Tempus and Zebra Medical Vision are gaining traction with their AI-powered healthcare platforms.

### Noteworthy News:
In a recent study published in Nature Medicine, researchers at Stanford University unveiled an AI algorithm capable of predicting patient outcomes based on electronic health record data. This breakthrough has the potential to transform how healthcare providers deliver personalized care to their patients.

## Target Audience Analysis

### Demographics:
The target audience for this blog post comprises healthcare professionals, researchers, and technology enthusiasts keen on exploring the intersection of AI and healthcare.

### Interests:
Our audience is particularly interested in staying abreast of the latest advancements in AI technology and its applications in enhancing patient outcomes within the healthcare sector.

### Pain Points:
Healthcare professionals often grapple with the overwhelming volume of data they need to analyze to make informed decisions. AI stands to alleviate this burden by automating certain tasks and offering valuable insights.

## Key Points

1. AI is reshaping the healthcare industry by enhancing diagnostics, tailoring treatment plans, and streamlining administrative tasks.
2. Key players in the AI healthcare sector include industry giants like IBM Watson Health, NVIDIA, as well as innovative startups like Tempus and Zebra Medical Vision.
3. The implementation of AI algorithms in healthcare has the potential to revolutionize patient outcomes and elevate the overall quality of care.

## Call to Action

As AI continues to advance in the healthcare sector, it is crucial for healthcare professionals to stay informed and adapt to these technological progressions. By embracing AI tools and platforms, healthcare providers can elevate patient care standards and drive superior outcomes.

### SEO Keywords and Relevant Data/Sources

#### Keywords:
1. Artificial Intelligence in Healthcare
2. AI Trends in Healthcare
3. Key Players in AI Healthcare

#### Sources:
1. [Nature Medicine Study](https://www.nature.com/articles/s41746-021-00470-x)
2. [IBM Watson Health](https://www.ibm.com/watson/health)
3. [NVIDIA Healthcare AI](https://www.nvidia.com/en-us/healthcare/ai-in-healthcare/)

<a name='1'></a>
 ## Other Popular Models as LLM for your Agents

#### Hugging Face (HuggingFaceHub endpoint)

```Python
from langchain_community.llms import HuggingFaceHub

llm = HuggingFaceHub(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    huggingfacehub_api_token="<HF_TOKEN_HERE>",
    task="text-generation",
)

### you will pass "llm" to your agent function
```

#### Mistral API

```Python
OPENAI_API_KEY=your-mistral-api-key
OPENAI_API_BASE=https://api.mistral.ai/v1
OPENAI_MODEL_NAME="mistral-small"
```

#### Cohere

```Python
from langchain_community.chat_models import ChatCohere
# Initialize language model
os.environ["COHERE_API_KEY"] = "your-cohere-api-key"
llm = ChatCohere()

### you will pass "llm" to your agent function
```

### For using Llama locally with Ollama and more, checkout the crewAI documentation on [Connecting to any LLM](https://docs.crewai.com/how-to/LLM-Connections/).